In [1]:
import json
import pandas as pd
from pathlib import Path
from emfdscore.scoring import score_docs

In [8]:
INPUT_JSON = Path("./processed/presidency_enriched.json")
OUTPUT_JSON = Path("./processed/presidency_enriched_emfd.json")

In [10]:
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    enriched_json = json.load(f)

print(type(enriched_json))

<class 'dict'>


In [11]:
print(len(enriched_json))

43


In [12]:
print(enriched_json[0].keys())

KeyError: 0

In [13]:
for k in enriched_json.keys():
    print(k)

Andrew Johnson
Millard Fillmore
Calvin Coolidge
James Monroe
Zachary Taylor
Franklin D. Roosevelt
Warren G. Harding
Theodore Roosevelt
James A. Garfield
George Washington
James Madison
Ronald Reagan
Martin van Buren
Richard Nixon
Herbert Hoover
Harry S. Truman
Benjamin Harrison
John Quincy Adams
Thomas Jefferson
Chester A. Arthur
Rutherford B. Hayes
James Buchanan
George Bush
Gerald R. Ford
John F. Kennedy
Dwight D. Eisenhower
Barack Obama
Lyndon B. Johnson
James K. Polk
Jimmy Carter
Abraham Lincoln
William Henry Harrison
William J. Clinton
John Adams
Franklin Pierce
Woodrow Wilson
William McKinley
George W. Bush
John Tyler
William Howard Taft
Grover Cleveland
Andrew Jackson
Ulysses S. Grant


In [14]:
president = list(enriched_json.keys())[0]

print(president)

print(type(enriched_json[president]))

print(len(enriched_json[president]))

print(enriched_json[president][0].keys())

Andrew Johnson
<class 'dict'>
6


KeyError: 0

In [15]:
president = list(enriched_json.keys())[0]

print(president)
print(enriched_json[president].keys())


Andrew Johnson
dict_keys(['OMB', 'E.O.P.', 'Party Platforms', 'Debates', 'Written', 'Oral'])


In [16]:
all_docs = []

def collect_documents(obj):
    if isinstance(obj, dict):
        if "content" in obj and "title" in obj:
            all_docs.append(obj)
        else:
            for v in obj.values():
                collect_documents(v)

    elif isinstance(obj, list):
        for item in obj:
            collect_documents(item)

collect_documents(enriched_json)

print(len(all_docs))
print(all_docs[0].keys())

58612
dict_keys(['category', 'subcategory', 'title', 'pid', 'content', 'document_date', 'speaker', 'party', 'document_type', 'next_election', 'next_election_date', 'days_before_next_election', 'speaker_role_next_election'])


In [17]:
import pandas as pd
from emfdscore.scoring import score_docs

In [18]:
texts = [
    str(doc.get("content", "") or "")
    for doc in all_docs
]

emfd_input = pd.DataFrame({0: texts})

print(emfd_input.shape)
print(emfd_input.head())

(58612, 1)
                                                   0
0  To the Senate and House of Representatives of ...
1  To the Senate of the United States: In answer ...
2  To the House of Representatives: In answer to ...
3  To the Senate and House of Representatives of ...
4  To the Senate of the United States: In reply t...


In [19]:
scores = score_docs(
    emfd_input,
    "emfd",
    "single",
    "bow",
    "sentiment",
    len(emfd_input)
)

print(scores.shape)
scores.head()

Processed: 0   0% |                      | Elapsed Time: 0:00:00 ETA:  --:--:--
Processed: 9   0% |                      | Elapsed Time: 0:00:00 ETA:   0:11:42
Processed: 25   0% |                     | Elapsed Time: 0:00:00 ETA:   0:08:10
Processed: 46   0% |                     | Elapsed Time: 0:00:00 ETA:   0:06:36
Processed: 61   0% |                     | Elapsed Time: 0:00:00 ETA:   0:06:40
Processed: 81   0% |                     | Elapsed Time: 0:00:00 ETA:   0:06:14
Processed: 95   0% |                     | Elapsed Time: 0:00:00 ETA:   0:06:22
Processed: 102   0% |                    | Elapsed Time: 0:00:00 ETA:   0:07:07
Processed: 117   0% |                    | Elapsed Time: 0:00:00 ETA:   0:07:02
Processed: 135   0% |                    | Elapsed Time: 0:00:00 ETA:   0:06:49
Processed: 153   0% |                    | Elapsed Time: 0:00:01 ETA:   0:06:41
Processed: 170   0% |                    | Elapsed Time: 0:00:01 ETA:   0:06:36
Processed: 188   0% |                   

(58612, 13)


,care_p,fairness_p,loyalty_p,authority_p,sanctity_p,care_sent,fairness_sent,loyalty_sent,authority_sent,sanctity_sent,moral_nonmoral_ratio,f_var,sent_var
0,0.019273,0.014612,0.003767,0.042003,0.029567,-0.030933,-0.006500,0.032227,-0.012930,0.003529,1.500000,0.000213,0.000544
1,0.000000,0.054750,0.000317,0.037792,0.000000,0.000000,-0.033995,0.019756,-0.024658,0.000000,0.846154,0.000675,0.000463
2,0.004181,0.030254,0.014815,0.045742,0.019608,-0.013895,-0.008402,0.010549,-0.017093,-0.002121,1.363636,0.000251,0.000120
3,0.040309,0.042440,0.021485,0.024107,0.000000,-0.098115,0.006171,-0.017272,0.017984,0.000000,1.700000,0.000294,0.002155
4,0.028274,0.031116,0.024926,0.052394,0.006503,-0.019273,-0.009408,0.018808,0.000756,0.008458,1.364407,0.000269,0.000221


In [20]:
EMFD_COLS = [
    "care_p", "fairness_p", "loyalty_p", "authority_p", "sanctity_p",
    "care_sent", "fairness_sent", "loyalty_sent", "authority_sent", "sanctity_sent",
    "moral_nonmoral_ratio", "f_var", "sent_var"
]

for i, doc in enumerate(all_docs):
    doc["emfdscore"] = {
        col: None if pd.isna(scores.loc[i, col]) else float(scores.loc[i, col])
        for col in EMFD_COLS
    }

In [21]:
import json

with open("presidency_enriched_emfd.json", "w", encoding="utf-8") as f:
    json.dump(enriched_json, f, ensure_ascii=False, indent=2)

In [23]:
print(all_docs[0]["title"])
print(all_docs[0]["emfdscore"])


Special Message
{'care_p': 0.019272807708791286, 'fairness_p': 0.014612268518518517, 'loyalty_p': 0.003766978759465555, 'authority_p': 0.04200331268725721, 'sanctity_p': 0.02956738250855898, 'care_sent': -0.0309328373015873, 'fairness_sent': -0.006499867091972351, 'loyalty_sent': 0.03222688888888889, 'authority_sent': -0.01293048975627744, 'sanctity_sent': 0.0035288888888888887, 'moral_nonmoral_ratio': 1.5, 'f_var': 0.00021293404770819462, 'sent_var': 0.0005436590906776359}


BURADAN SONRASI DENEMELER

In [ ]:
cols = [
    "care_p",
    "fairness_p",
    "loyalty_p",
    "authority_p",
    "sanctity_p"
]

df[cols].corr()

NameError: name 'df' is not defined

In [25]:
import pandas as pd

rows = []

for doc in all_docs:
    rows.append({
        "speaker": doc.get("speaker"),
        "document_type": doc.get("document_type"),

        "care_p": doc["emfdscore"]["care_p"],
        "fairness_p": doc["emfdscore"]["fairness_p"],
        "loyalty_p": doc["emfdscore"]["loyalty_p"],
        "authority_p": doc["emfdscore"]["authority_p"],
        "sanctity_p": doc["emfdscore"]["sanctity_p"],
    })

df = pd.DataFrame(rows)

print(df.shape)
df.head()

(58612, 7)


,speaker,document_type,care_p,fairness_p,loyalty_p,authority_p,sanctity_p
0,Andrew Johnson,Written,0.019273,0.014612,0.003767,0.042003,0.029567
1,Andrew Johnson,Written,0.000000,0.054750,0.000317,0.037792,0.000000
2,Andrew Johnson,Written,0.004181,0.030254,0.014815,0.045742,0.019608
3,Andrew Johnson,Written,0.040309,0.042440,0.021485,0.024107,0.000000
4,Andrew Johnson,Written,0.028274,0.031116,0.024926,0.052394,0.006503


In [26]:
cols = [
    "care_p",
    "fairness_p",
    "loyalty_p",
    "authority_p",
    "sanctity_p"
]

corr = df[cols].corr()

corr

,care_p,fairness_p,loyalty_p,authority_p,sanctity_p
care_p,1.000000,-0.130851,0.128193,-0.502821,0.159618
fairness_p,-0.130851,1.000000,-0.180714,-0.159196,-0.068849
loyalty_p,0.128193,-0.180714,1.000000,-0.462084,0.104894
authority_p,-0.502821,-0.159196,-0.462084,1.000000,-0.257835
sanctity_p,0.159618,-0.068849,0.104894,-0.257835,1.000000


In [27]:
corr.round(3)

,care_p,fairness_p,loyalty_p,authority_p,sanctity_p
care_p,1.000,-0.131,0.128,-0.503,0.160
fairness_p,-0.131,1.000,-0.181,-0.159,-0.069
loyalty_p,0.128,-0.181,1.000,-0.462,0.105
authority_p,-0.503,-0.159,-0.462,1.000,-0.258
sanctity_p,0.160,-0.069,0.105,-0.258,1.000


In [28]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd

cols = [
    "care_p",
    "fairness_p",
    "loyalty_p",
    "authority_p",
    "sanctity_p"
]

X = df[cols].fillna(0)

X_std = StandardScaler().fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_std)

explained = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(cols))],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative": pca.explained_variance_ratio_.cumsum()
})

explained

,PC,explained_variance_ratio,cumulative
0,PC1,0.373930,0.373930
1,PC2,0.220464,0.594393
2,PC3,0.183183,0.777577
3,PC4,0.168346,0.945923
4,PC5,0.054077,1.000000


In [29]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=cols,
    columns=[f"PC{i+1}" for i in range(len(cols))]
)

loadings.round(3)

,PC1,PC2,PC3,PC4,PC5
care_p,-0.509,0.011,0.180,0.718,0.440
fairness_p,0.086,0.919,-0.061,-0.151,0.349
loyalty_p,-0.469,-0.233,-0.582,-0.448,0.432
authority_p,0.629,-0.315,0.144,-0.011,0.695
sanctity_p,-0.343,-0.046,0.777,-0.511,0.120


Bu PCA sonucu net:

PC1 = Authority karşısında Care + Loyalty + Sanctity

Çünkü:

authority_p   +0.629
care_p        -0.509
loyalty_p     -0.469
sanctity_p    -0.343

Bu eksen şuna benziyor:

idari/kurumsal otorite dili  ↔  toplumsal-mobilize edici moral dil

PC1 tek başına varyansın %37.4’ünü açıklıyor.

PC2 = Fairness ekseni

Çünkü:

fairness_p +0.919

Bu neredeyse bağımsız bir “justice/fairness” boyutu.

PC3 = Sanctity karşısında Loyalty
sanctity_p +0.777
loyalty_p  -0.582

Bu da “saflık/kutsallık dili ↔ grup/sadakat dili” ayrımı gibi.

En önemli sonuç:

Senin eski varsayım:

toplumsal_yasa = (authority + loyalty) / 2

bu corpus için geometrik olarak zayıf.

Çünkü PCA’da:

authority

ve

loyalty

aynı yöne değil, karşıt yöne yakın duruyor.

Bu yüzden şimdilik üç ekseni JSON’a eklememek doğru olmuş.

Şu aşamada yapılacak en sağlıklı şey:

emfdscore ham skorları JSON’da sakla.
PCA skorlarını ayrı analiz DataFrame’inde üret.
Kendi teorik eksenlerini doğrudan ortalama olarak değil, covariance-aware dönüşümle kur.

In [30]:
pca_cols = [f"PC{i+1}" for i in range(5)]

df_pca = pd.DataFrame(
    X_pca,
    columns=pca_cols
)

df_with_pca = pd.concat(
    [df.reset_index(drop=True), df_pca],
    axis=1
)

df_with_pca.head()

,speaker,document_type,care_p,fairness_p,loyalty_p,authority_p,sanctity_p,PC1,PC2,PC3,PC4,PC5
0,Andrew Johnson,Written,0.019273,0.014612,0.003767,0.042003,0.029567,0.604812,-1.155874,3.387852,-1.524643,-1.043267
1,Andrew Johnson,Written,0.000000,0.054750,0.000317,0.037792,0.000000,2.977951,1.846684,-0.351215,-0.353246,-1.457998
2,Andrew Johnson,Written,0.004181,0.030254,0.014815,0.045742,0.019608,1.352074,-0.403533,1.377802,-2.074753,-0.594318
3,Andrew Johnson,Written,0.040309,0.042440,0.021485,0.024107,0.000000,0.060527,0.882039,-1.198065,0.645258,-0.438156
4,Andrew Johnson,Written,0.028274,0.031116,0.024926,0.052394,0.006503,1.113863,-0.616951,-0.422557,-0.445401,0.592583


In [ ]:
df_with_pca.sort_values("PC1", ascending=False).head(20)


In [ ]:
df_with_pca.sort_values("PC1", ascending=True).head(20)

In [42]:
df_with_pca.sort_values("PC5", ascending=False).head(20).to_excel("pc5_highest_20.xlsx", index=False)

In [43]:
df_with_pca.sort_values("PC5", ascending=True).head(20).to_excel("pc5_lowest_20.xlsx", index=False)

In [44]:
# Eğer df_with_pca içinde doc_idx yoksa ekle
df_with_pca = df_with_pca.reset_index(drop=True)
df_with_pca["doc_idx"] = range(len(df_with_pca))

assert len(df_with_pca) == len(all_docs)

In [45]:
def export_pc_extremes(df_with_pca, all_docs, pcs=None, n=20, max_chars=20_000):
    if pcs is None:
        pcs = ["PC1", "PC2", "PC3", "PC4", "PC5"]

    rows = []

    for pc in pcs:
        for direction, ascending in [("HIGH", False), ("LOW", True)]:

            subset = (
                df_with_pca
                .sort_values(pc, ascending=ascending)
                .head(n)
            )

            for rank, (_, row) in enumerate(subset.iterrows(), start=1):
                doc_idx = int(row["doc_idx"])
                doc = all_docs[doc_idx]

                text = doc.get("content", "") or ""

                rows.append({
                    "pc": pc,
                    "direction": direction,
                    "rank": rank,
                    "pc_score": float(row[pc]),

                    "doc_idx": doc_idx,
                    "speaker": doc.get("speaker"),
                    "party": doc.get("party"),
                    "title": doc.get("title"),
                    "document_date": doc.get("document_date"),
                    "document_type": doc.get("document_type"),
                    "category": doc.get("category"),
                    "subcategory": doc.get("subcategory"),
                    "pid": doc.get("pid"),

                    "care_p": doc["emfdscore"]["care_p"],
                    "fairness_p": doc["emfdscore"]["fairness_p"],
                    "loyalty_p": doc["emfdscore"]["loyalty_p"],
                    "authority_p": doc["emfdscore"]["authority_p"],
                    "sanctity_p": doc["emfdscore"]["sanctity_p"],

                    "text": text[:max_chars],
                    "text_char_count_original": len(text),
                    "text_was_truncated": len(text) > max_chars,
                })

    return pd.DataFrame(rows)

In [46]:
pc_extremes = export_pc_extremes(
    df_with_pca=df_with_pca,
    all_docs=all_docs,
    pcs=["PC1", "PC2", "PC3", "PC4", "PC5"],
    n=20,
    max_chars=20_000
)

print(pc_extremes.shape)
pc_extremes.head()

(200, 21)


,pc,direction,rank,pc_score,doc_idx,speaker,party,title,document_date,document_type,...,subcategory,pid,care_p,fairness_p,loyalty_p,authority_p,sanctity_p,text,text_char_count_original,text_was_truncated
0,PC1,HIGH,1,6.358194,41514,Abraham Lincoln,Republican,Executive Order,1861-11-27 00:00:00,Written,...,None,70089,0.000955,0.000000,0.000000,0.122674,0.0,The municipal authorities of Washington and Ge...,294,False
1,PC1,HIGH,2,6.275333,57913,Andrew Jackson,Democratic,Special Message,1833-02-12 00:00:00,Written,...,to Congress,66907,0.000000,0.037229,0.000000,0.114904,0.0,To the Senate. I transmit herewith to the Sena...,371,False
2,PC1,HIGH,3,6.251471,6,Andrew Johnson,National Union,Special Message,1865-12-21 00:00:00,Written,...,to Congress,72369,0.000000,0.025866,0.000000,0.115950,0.0,To the Senate: In compliance with the resoluti...,309,False
3,PC1,HIGH,4,6.135593,9719,Martin van Buren,None,Special Message,1838-01-13 00:00:00,Written,...,to Congress,67229,0.000000,0.060150,0.000000,0.108523,0.0,"To the Senate: I transmit to the Senate, for i...",166,False
4,PC1,HIGH,5,5.948666,15712,Harry S. Truman,Democratic,100 - Statement by the President Announcing R...,1948-05-14 00:00:00,Written,...,(non categorized),12896,0.013304,0.000000,0.000317,0.122511,0.0,THIS GOVERNMENT has been informed that a Jewis...,271,False


In [47]:
assert len(pc_extremes) == 5 * 2 * 20

In [48]:
pc_extremes.to_excel("pc_high_low_20_texts_for_analysis.xlsx", index=False)
pc_extremes.to_csv("pc_high_low_20_texts_for_analysis.csv", index=False, encoding="utf-8-sig")

In [49]:
pc_extremes.groupby(["pc", "direction"]).size()

pc   direction
PC1  HIGH         20
     LOW          20
PC2  HIGH         20
     LOW          20
PC3  HIGH         20
     LOW          20
PC4  HIGH         20
     LOW          20
PC5  HIGH         20
     LOW          20
dtype: int64

In [50]:
pc_extremes[
    (pc_extremes["pc"] == "PC5") &
    (pc_extremes["direction"] == "HIGH")
][["rank", "speaker", "title", "document_date", "pc_score"]]

,rank,speaker,title,document_date,pc_score
160,1,Herbert Hoover,466 - Message to President-Elect Roosevelt Ab...,1933-02-15 00:00:00,6.217005
161,2,George W. Bush,Remarks Opening the 2002 Olympic Winter Games ...,2002-02-08 00:00:00,3.168804
162,3,Franklin D. Roosevelt,121 - Executive Order 7139 Extending the Juri...,1935-08-12 00:00:00,2.944923
163,4,Barack Obama,Proclamation 8535—Flag Day and National Flag W...,2010-06-11 00:00:00,2.752775
164,5,Ronald Reagan,Proclamation 4846—Flag Day and National Flag W...,1981-06-01 00:00:00,2.693395
165,6,Barack Obama,Proclamation 8689—Flag Day and National Flag W...,2011-06-10 00:00:00,2.595257
166,7,George W. Bush,Proclamation 8155—Flag Day and National Flag W...,2007-06-05 00:00:00,2.548459
167,8,Lyndon B. Johnson,768 - The President's Toast to Gustavo Diaz O...,1964-11-12 00:00:00,2.525979
168,9,George W. Bush,Proclamation 8269—Flag Day and National Flag W...,2008-06-06 00:00:00,2.506978
169,10,Abraham Lincoln,Executive Order,1861-10-04 00:00:00,2.418600
